In [3]:
import sqlite3
import pandas as pd
import numpy as np
import joblib

DB_NAME = "cat_fleet_medallion.db"
MODEL_PATH = "cat_fleet_failure_model.pkl"
FEATURES_PATH = "model_features.pkl"

In [6]:
model = joblib.load(MODEL_PATH)
feature_cols = joblib.load(FEATURES_PATH)

print(f"Model loaded: {type(model).__name__}")
print(f"Model expects {len(feature_cols)} features")
print(f"First 10 expected features: {feature_cols[:10]}")

Model loaded: XGBClassifier
Model expects 41 features
First 10 expected features: ['ODOMETER_READING', 'ENGINE_TEMP_C', 'ENGINE_RPM', 'OIL_PRESSURE_PSI', 'COOLANT_TEMP_C', 'FUEL_LEVEL_PERCENT', 'FUEL_CONSUMPTION_LPH', 'ENGINE_LOAD_PERCENT', 'THROTTLE_POS_PERCENT', 'AIR_FLOW_RATE_GPS']


In [11]:
conn = sqlite3.connect(DB_NAME)
df_gold=pd.read_sql("SELECT * FROM gold_telematics",conn)
conn.close()

df_gold.columns=[c.upper() for c in df_gold.columns]
print(f"gold table loaded:{df_gold.shape}")
print(f"columns: {df_gold.columns.tolist()}")

gold table loaded:(1970, 56)
columns: ['MACHINE_ID', 'BRAND', 'TIMESTAMP', 'ODOMETER_READING', 'ENGINE_TEMP_C', 'ENGINE_RPM', 'OIL_PRESSURE_PSI', 'COOLANT_TEMP_C', 'FUEL_LEVEL_PERCENT', 'FUEL_CONSUMPTION_LPH', 'ENGINE_LOAD_PERCENT', 'THROTTLE_POS_PERCENT', 'AIR_FLOW_RATE_GPS', 'EXHAUST_GAS_TEMP_C', 'VIBRATION_LEVEL', 'ENGINE_HOURS', 'BRAKE_FLUID_LEVEL_PSI', 'BRAKE_PAD_WEAR_MM', 'BRAKE_TEMP_C', 'ABS_FAULT_INDICATOR', 'BRAKE_PEDAL_POS_PERCENT', 'WHEEL_SPEED_FL_KPH', 'WHEEL_SPEED_FR_KPH', 'WHEEL_SPEED_RL_KPH', 'WHEEL_SPEED_RR_KPH', 'BATTERY_VOLTAGE_V', 'BATTERY_CURRENT_A', 'BATTERY_TEMP_C', 'ALTERNATOR_OUTPUT_V', 'BATTERY_CHARGE_PERCENT', 'BATTERY_HEALTH_PERCENT', 'VEHICLE_SPEED_KPH', 'AMBIENT_TEMP_C', 'HUMIDITY_PERCENT', 'GPS_LATITUDE', 'GPS_LONGITUDE', 'ENGINE_FAILURE_IMMINENT', 'BRAKE_ISSUE_IMMINENT', 'BATTERY_ISSUE_IMMINENT', 'FAILURE_DATE', 'FAILURE_TYPE', 'CLEAN_FAILURE_TYPE', 'IS_IDLING', 'FAILURE_TIMESTAMP', 'HOURS_UNTIL_FAILURE', 'FINAL_TARGET_LABEL', 'ENGINE_TEMP_C_ROLL3_MEAN', 

In [13]:
df_gold['TIMESTAMP'] = pd.to_datetime(df_gold['TIMESTAMP'])

new_data = (
    df_gold.sort_values(['MACHINE_ID', 'TIMESTAMP'])
    .groupby('MACHINE_ID')
    .tail(5)
    .reset_index(drop=True)
)

print(f"Simulated new data: {len(new_data)} rows across {new_data['MACHINE_ID'].nunique()} machines")
new_data[['MACHINE_ID', 'TIMESTAMP']].head(10)

Simulated new data: 250 rows across 50 machines


,MACHINE_ID,TIMESTAMP
0,VEH0000,2023-01-02 01:00:00
1,VEH0000,2023-01-02 04:03:00
2,VEH0000,2023-01-02 08:05:00
3,VEH0000,2023-01-02 09:26:00
4,VEH0000,2023-01-02 10:32:00
5,VEH0001,2023-01-02 04:00:00
6,VEH0001,2023-01-02 04:30:00
7,VEH0001,2023-01-02 04:36:00
8,VEH0001,2023-01-02 10:48:00
9,VEH0001,2023-01-02 14:57:00


In [17]:
available_cols = set(new_data.columns)
expected_cols = set(feature_cols)

missing_in_new_data = expected_cols - available_cols
extra_in_new_data = available_cols - expected_cols

if missing_in_new_data:
    print(f"MISSING columns the model needs but new_data doesn't have:")
    print(f"{sorted(missing_in_new_data)}")
else:
    print("All expected feature columns are present in new_data.")

print(f"\nExtra columns in new_data (not used by model, informational only):")
print(f"{sorted(list(extra_in_new_data))[:10]}")

All expected feature columns are present in new_data.

Extra columns in new_data (not used by model, informational only):
['BATTERY_ISSUE_IMMINENT', 'BRAKE_ISSUE_IMMINENT', 'BRAND', 'CLEAN_FAILURE_TYPE', 'ENGINE_FAILURE_IMMINENT', 'FAILURE_DATE', 'FAILURE_TIMESTAMP', 'FAILURE_TYPE', 'FINAL_TARGET_LABEL', 'GPS_LATITUDE']


In [18]:
assert not missing_in_new_data, "Column mismatch detected - fix names before processing."
X_new=new_data[feature_cols]
print(f"X_new shape:{X_new.shape}")

X_new shape:(250, 41)


In [21]:
nan_counts = X_new.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
print(nan_cols if len(nan_cols) > 0 else "No NaN values in X_new.")

No NaN values in X_new.


In [23]:
predictions = model.predict(X_new)
probabilities = model.predict_proba(X_new)[:, 1]

print(f"Predicted 'At Risk': {(predictions == 1).sum()}")
print(f"Predicted 'Healthy': {(predictions == 0).sum()}")

Predicted 'At Risk': 3
Predicted 'Healthy': 247


In [29]:
results = new_data[['MACHINE_ID', 'TIMESTAMP']].copy()
results['PREDICTED_LABEL'] = np.where(predictions == 1, 'At Risk', 'Healthy')
results['FAILURE_PROBABILITY'] = probabilities.round(4)

results = results.sort_values('FAILURE_PROBABILITY', ascending=False).reset_index(drop=True)
results.head(20)

,MACHINE_ID,TIMESTAMP,PREDICTED_LABEL,FAILURE_PROBABILITY
0,VEH0025,2023-01-02 04:00:00,At Risk,0.9994
1,VEH0036,2023-01-02 10:25:00,At Risk,0.9992
2,VEH0047,2023-01-02 04:03:00,At Risk,0.9992
3,VEH0035,2023-01-02 06:29:00,Healthy,0.0909
4,VEH0028,2023-01-02 00:10:00,Healthy,0.0877
5,VEH0026,2023-01-01 16:55:00,Healthy,0.0314
6,VEH0011,2023-01-01 23:12:00,Healthy,0.0291
7,VEH0034,2023-01-02 00:04:00,Healthy,0.0077
8,VEH0007,2023-01-02 09:26:00,Healthy,0.0041
9,VEH0014,2023-01-01 21:28:00,Healthy,0.0033


In [30]:
results.to_csv('inference_predictions.csv', index=False)
print(f"Saved {len(results)} predictions to inference_predictions.csv")

Saved 250 predictions to inference_predictions.csv
